# Gitter-Welt – Ein Agent lernt, den besten Weg zu finden

**Worum geht's?** Ein Computer-Agent steht auf einem Gitter und muss zu einem Ziel finden – dabei gibt es Fallen, die er meiden soll. Anfangs läuft er komplett planlos umher. Durch **Reinforcement Learning** (Lernen durch Belohnung) findet er nach und nach den besten Weg – ganz von allein!

---

## Die Spielwelt

Wir haben ein Gitter mit 5×5 Feldern:

- 🟢 **Start** oben links
- 🏁 **Ziel** unten rechts – dafür gibt es Punkte!
- 🔥 **Fallen** – wer hineinläuft, verliert Punkte und das Spiel ist sofort vorbei
- ⬛ **Wand** – hier kann man nicht durchlaufen

Der Agent kann sich in vier Richtungen bewegen: **hoch, runter, links, rechts**.

**Der wichtige Unterschied zu unseren vorigen Spielen:** Hier ist eine einzelne Entscheidung nicht sofort gut oder schlecht – ein Schritt nach rechts ist nur dann clever, wenn er auf einem *guten Weg* zum Ziel liegt. Der Agent muss lernen, **mehrere Schritte vorauszudenken**.

---
## Teil 1: Die Spielwelt bauen

In [ ]:
# Diese Zelle zuerst ausführen - sie lädt alles, was wir brauchen
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import defaultdict

print("✅ Bereit zum Spielen!")

In [ ]:
# ── Die Spielwelt ──
GROESSE = 5                      # 5x5 Gitter
START = (0, 0)                   # oben links
ZIEL = (4, 4)                    # unten rechts
FALLEN = {(1, 3), (3, 1)}        # zwei Fallen-Felder
WAND = {(2, 2)}                  # ein blockiertes Feld

AKTIONEN = ["hoch", "runter", "links", "rechts"]
BEWEGUNG = {"hoch": (-1, 0), "runter": (1, 0), "links": (0, -1), "rechts": (0, 1)}


def schritt(position, aktion):
    """
    Bewegt den Agenten um ein Feld. Gibt zurück:
        neue_position, belohnung, ist_das_spiel_zu_ende
    """
    dy, dx = BEWEGUNG[aktion]
    neue_position = (position[0] + dy, position[1] + dx)

    # Außerhalb des Gitters oder gegen die Wand? -> Agent bleibt stehen
    if not (0 <= neue_position[0] < GROESSE and 0 <= neue_position[1] < GROESSE):
        neue_position = position
    if neue_position in WAND:
        neue_position = position

    if neue_position == ZIEL:
        return neue_position, 10, True       # Ziel erreicht: großer Bonus, Spiel zu Ende
    elif neue_position in FALLEN:
        return neue_position, -10, True      # Falle: großer Verlust, Spiel zu Ende
    else:
        return neue_position, -1, False      # Jeder normale Schritt kostet 1 Punkt


print("✅ Spielwelt definiert")
print(f"Start: {START}  |  Ziel: {ZIEL}  |  Fallen: {FALLEN}  |  Wand: {WAND}")

**Wieso kostet jeder normale Schritt -1 Punkt?** Das ist ein kleiner Trick: So lernt der Agent nicht nur "vermeide Fallen", sondern auch "finde den *kürzesten* Weg" – denn jeder unnötige Umweg kostet ihn Punkte!

In [ ]:
def zeige_gitter(position=None, pfad=None, titel="Die Spielwelt"):
    """
    Zeichnet das Gitter. Optional: aktuelle Position des Agenten und/oder einen Pfad.
    """
    fig, ax = plt.subplots(figsize=(5.5, 5.5))

    for zeile in range(GROESSE):
        for spalte in range(GROESSE):
            feld = (zeile, spalte)
            if feld == ZIEL:
                farbe = "#A5D6A7"   # grün
            elif feld in FALLEN:
                farbe = "#EF9A9A"   # rot
            elif feld in WAND:
                farbe = "#616161"   # grau
            elif feld == START:
                farbe = "#90CAF9"   # blau
            else:
                farbe = "#FAFAFA"   # weiß
            rect = mpatches.Rectangle((spalte, GROESSE - 1 - zeile), 1, 1,
                                       facecolor=farbe, edgecolor="#9E9E9E", linewidth=1)
            ax.add_patch(rect)

            # Beschriftung
            text = ""
            if feld == ZIEL: text = "ZIEL"
            elif feld in FALLEN: text = "FALLE"
            elif feld in WAND: text = "WAND"
            elif feld == START: text = "START"
            if text:
                ax.text(spalte + 0.5, GROESSE - 1 - zeile + 0.5, text,
                        ha="center", va="center", fontsize=9, fontweight="bold")

    # Pfad als Linie einzeichnen
    if pfad and len(pfad) > 1:
        xs = [p[1] + 0.5 for p in pfad]
        ys = [GROESSE - 1 - p[0] + 0.5 for p in pfad]
        ax.plot(xs, ys, color="#1565C0", linewidth=2.5, marker="o", markersize=6, zorder=5)

    # Aktuelle Position als Punkt
    if position:
        ax.plot(position[1] + 0.5, GROESSE - 1 - position[0] + 0.5,
                marker="o", markersize=20, color="#FF6F00", zorder=6)

    ax.set_xlim(0, GROESSE)
    ax.set_ylim(0, GROESSE)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(titel)
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()


zeige_gitter()

---
## Teil 2: Ein planloser Agent

Bevor wir trainieren, schauen wir uns an, wie sich ein Agent verhält, der **rein zufällig** läuft – ohne irgendetwas gelernt zu haben.

In [ ]:
def zufalls_versuch(max_schritte=30):
    """Lässt einen Agenten rein zufällig laufen und zeigt den Weg."""
    position = START
    pfad = [position]
    gesamt_belohnung = 0

    for schritt_nr in range(max_schritte):
        aktion = random.choice(AKTIONEN)
        position, belohnung, fertig = schritt(position, aktion)
        pfad.append(position)
        gesamt_belohnung += belohnung
        if fertig:
            break

    ergebnis = "Ziel erreicht! 🎉" if position == ZIEL else ("In Falle gelaufen 🔥" if position in FALLEN else "Zeit abgelaufen ⏱️")
    ergebnis_titel = "Ziel erreicht!" if position == ZIEL else ("In Falle gelaufen" if position in FALLEN else "Zeit abgelaufen")
    print(f"Ergebnis: {ergebnis}")
    print(f"Anzahl Schritte: {len(pfad) - 1}")
    print(f"Gesamtpunktzahl: {gesamt_belohnung}")
    zeige_gitter(pfad=pfad, titel=f"Zufälliger Versuch ({ergebnis_titel})")


zufalls_versuch()

**Führt die Zelle oben mehrmals aus** (Strg+Enter oder ▶️) – ihr werdet sehen: Ohne Lernen findet der Agent das Ziel nur durch puren Zufall, läuft oft im Kreis oder direkt in eine Falle.

---
## Teil 3: Der lernende Agent

### Was ist neu gegenüber unseren vorigen Spielen?

Bei Schere-Stein-Papier hat der Agent direkt nach jeder Entscheidung sein Feedback bekommen. Hier ist es anders: Eine Entscheidung in der **Mitte** des Weges (z.B. "nach rechts") wirkt sich erst **später** aus – nämlich ob man am Ende ins Ziel oder in eine Falle läuft.

**Die Lösung:** Der Agent denkt nicht nur an die Belohnung *dieses* Schritts, sondern auch ein bisschen an die Punktzahl, die er sich für das *nächste* Feld bereits erarbeitet hat. So "fließt" die Information vom Ziel rückwärts durch das ganze Gitter – Schritt für Schritt, Spiel für Spiel.

Das steuern wir mit einem neuen Wert: dem **Weitblick** (in Fachliteratur "Diskontfaktor" genannt). Ein hoher Weitblick (nah an 1) heißt: "Zukünftige Belohnungen sind mir fast genauso wichtig wie die jetzige." Ein niedriger Weitblick heißt: "Mir ist nur wichtig, was *gleich* passiert."

In [ ]:
class GitterAgent:
    """
    Ein Q-Learning Agent für die Gitter-Welt.
    
    Merkt sich für jedes Feld eine Punktzahl pro möglicher Richtung.
    """

    def __init__(self, lerntempo=0.1, weitblick=0.9, zufallsquote=1.0,
                 zufallsquote_minimum=0.05, abkuehlung=0.999):
        self.punkte = defaultdict(float)         # Punktzahl pro (Feld, Richtung)
        self.lerntempo = lerntempo                # Wie stark wirkt jede neue Erfahrung?
        self.weitblick = weitblick                # Wie wichtig ist die Zukunft? (0 - 1)
        self.zufallsquote = zufallsquote           # Wie oft wird ausprobiert?
        self.zufallsquote_minimum = zufallsquote_minimum
        self.abkuehlung = abkuehlung

    def waehle_richtung(self, position):
        """Meistens die bisher beste Richtung, manchmal zufällig zum Ausprobieren."""
        if random.random() < self.zufallsquote:
            return random.choice(AKTIONEN)
        return max(AKTIONEN, key=lambda a: self.punkte[(position, a)])

    def lerne(self, position, aktion, belohnung, neue_position, spiel_zu_ende):
        """Passt die Punktzahl nach einem Schritt an."""
        alte_punktzahl = self.punkte[(position, aktion)]

        if spiel_zu_ende:
            ziel_wert = belohnung
        else:
            beste_zukunft = max(self.punkte[(neue_position, a)] for a in AKTIONEN)
            ziel_wert = belohnung + self.weitblick * beste_zukunft

        self.punkte[(position, aktion)] = alte_punktzahl + self.lerntempo * (ziel_wert - alte_punktzahl)

        if self.zufallsquote > self.zufallsquote_minimum:
            self.zufallsquote *= self.abkuehlung


print("✅ Lernender Agent ist bereit")

---
## Teil 4: Training – viele Durchläufe üben

Der Agent spielt jetzt **5.000 komplette Durchläufe** (von Start bis Ziel oder Falle). Jeder Durchlauf nennt sich eine **Episode**.

In [ ]:
def trainiere_agent(anzahl_episoden=5000, max_schritte_pro_episode=50):
    """
    Lässt den Agenten viele Durchläufe spielen und lernen.
    Gibt den trainierten Agenten und eine Lern-Statistik zurück.
    """
    agent = GitterAgent()

    mess_intervall = 50
    erfolge_im_intervall = 0
    schritte_im_intervall = []
    verlauf_erfolgsquote = []
    verlauf_schritte = []
    verlauf_zufallsquote = []

    for episode in range(1, anzahl_episoden + 1):
        position = START
        anzahl_schritte = 0

        for _ in range(max_schritte_pro_episode):
            aktion = agent.waehle_richtung(position)
            neue_position, belohnung, fertig = schritt(position, aktion)
            agent.lerne(position, aktion, belohnung, neue_position, fertig)
            position = neue_position
            anzahl_schritte += 1
            if fertig:
                break

        if position == ZIEL:
            erfolge_im_intervall += 1
        schritte_im_intervall.append(anzahl_schritte)

        if episode % mess_intervall == 0:
            verlauf_erfolgsquote.append(erfolge_im_intervall / mess_intervall * 100)
            verlauf_schritte.append(sum(schritte_im_intervall) / len(schritte_im_intervall))
            verlauf_zufallsquote.append(agent.zufallsquote)
            erfolge_im_intervall = 0
            schritte_im_intervall = []

    statistik = {
        "erfolgsquote": verlauf_erfolgsquote,
        "schritte": verlauf_schritte,
        "zufallsquote": verlauf_zufallsquote,
        "intervall": mess_intervall,
    }
    return agent, statistik


print("Training läuft ...")
agent, statistik = trainiere_agent(anzahl_episoden=5000)
print("✅ Training abgeschlossen!")

---
## Teil 5: Wird der Agent besser?

In [ ]:
def zeige_lernkurve(statistik):
    episoden_achse = [i * statistik["intervall"] for i in range(1, len(statistik["erfolgsquote"]) + 1)]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Plot 1: Erfolgsquote
    ax = axes[0]
    ax.plot(episoden_achse, statistik["erfolgsquote"], color="#1565C0", linewidth=2)
    ax.set_xlabel("Trainings-Episode")
    ax.set_ylabel("Erfolgsquote (%)")
    ax.set_title("Wie oft erreicht der Agent das Ziel?")
    ax.set_ylim(0, 105)
    ax.grid(alpha=0.3)

    # Plot 2: Anzahl Schritte bis zum Ziel
    ax = axes[1]
    ax.plot(episoden_achse, statistik["schritte"], color="#E65100", linewidth=2)
    ax.axhline(y=8, color="gray", linestyle="--", alpha=0.7, label="Kürzester möglicher Weg (8)")
    ax.set_xlabel("Trainings-Episode")
    ax.set_ylabel("Durchschnittliche Schrittzahl")
    ax.set_title("Wird der Weg kürzer?")
    ax.legend()
    ax.grid(alpha=0.3)

    # Plot 3: Zufallsquote
    ax = axes[2]
    werte = [z * 100 for z in statistik["zufallsquote"]]
    ax.plot(episoden_achse, werte, color="#2E7D32", linewidth=2)
    ax.fill_between(episoden_achse, werte, alpha=0.2, color="#2E7D32")
    ax.set_xlabel("Trainings-Episode")
    ax.set_ylabel("Wie oft wird ausprobiert (%)")
    ax.set_title("Ausprobieren wird seltener")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

zeige_lernkurve(statistik)

**Was seht ihr?** Die Erfolgsquote steigt schnell auf nahezu 100%, und die Anzahl der Schritte nähert sich dem kürzestmöglichen Weg von 8 Feldern an. Der Agent hat nicht nur gelernt, *ob* er ankommt, sondern auch den **effizientesten** Weg gefunden!

---
## Teil 6: Den gelernten Weg anschauen

In [ ]:
def zeige_gelernten_weg(agent, max_schritte=30):
    """Lässt den trainierten Agenten den Weg greedy (ohne Zufall) ablaufen."""
    position = START
    pfad = [position]
    gesamt_belohnung = 0

    for _ in range(max_schritte):
        aktion = max(AKTIONEN, key=lambda a: agent.punkte[(position, a)])
        position, belohnung, fertig = schritt(position, aktion)
        pfad.append(position)
        gesamt_belohnung += belohnung
        if fertig:
            break

    ergebnis = "Ziel erreicht! 🎉" if position == ZIEL else ("In Falle gelaufen 🔥" if position in FALLEN else "Zeit abgelaufen ⏱️")
    print(f"Ergebnis: {ergebnis}")
    print(f"Anzahl Schritte: {len(pfad) - 1}")
    print(f"Gesamtpunktzahl: {gesamt_belohnung}")
    zeige_gitter(pfad=pfad, titel=f"Weg des trainierten Agenten ({len(pfad)-1} Schritte)")


zeige_gelernten_weg(agent)

---
## Teil 7: Was hat der Agent für jedes Feld gelernt?

Wir können uns sogar ansehen, welche Richtung der Agent von **jedem einzelnen Feld** aus für die beste hält – nicht nur entlang seines tatsächlichen Weges. Das nennt man die **Strategie** des Agenten.

In [ ]:
PFEIL = {"hoch": "↑", "runter": "↓", "links": "←", "rechts": "→"}


def zeige_strategie(agent):
    """Zeigt für jedes Feld die vom Agenten bevorzugte Richtung als Pfeil."""
    fig, ax = plt.subplots(figsize=(5.5, 5.5))

    for zeile in range(GROESSE):
        for spalte in range(GROESSE):
            feld = (zeile, spalte)
            if feld == ZIEL:
                farbe, text = "#A5D6A7", "ZIEL"
            elif feld in FALLEN:
                farbe, text = "#EF9A9A", "FALLE"
            elif feld in WAND:
                farbe, text = "#616161", "WAND"
            else:
                beste_richtung = max(AKTIONEN, key=lambda a: agent.punkte[(feld, a)])
                farbe, text = "#FAFAFA", PFEIL[beste_richtung]

            rect = mpatches.Rectangle((spalte, GROESSE - 1 - zeile), 1, 1,
                                       facecolor=farbe, edgecolor="#9E9E9E", linewidth=1)
            ax.add_patch(rect)
            fontsize = 22 if text in PFEIL.values() else 9
            fontweight = "normal" if text in PFEIL.values() else "bold"
            ax.text(spalte + 0.5, GROESSE - 1 - zeile + 0.5, text,
                    ha="center", va="center", fontsize=fontsize, fontweight=fontweight)

    ax.set_xlim(0, GROESSE)
    ax.set_ylim(0, GROESSE)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title("Gelernte Strategie: beste Richtung pro Feld")
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()

zeige_strategie(agent)

**Schaut euch die Pfeile in der Nähe der Fallen an:** Der Agent hat gelernt, einen Bogen um sie zu machen, obwohl das manchmal einen kleinen Umweg bedeutet. Das zeigt: Der Agent wägt ab zwischen "kürzester Weg" und "Risiko vermeiden".

---
## Teil 8: Selbst ausprobieren

### 🔬 Aufgaben

1. **Mehr Fallen:** Füge in `FALLEN` ein drittes Fallen-Feld hinzu, z.B. `(2, 3)`. Trainiere neu (Teil 4 erneut ausführen) – findet der Agent trotzdem einen Weg?

2. **Größeres Gitter:** Ändere `GROESSE = 5` auf `GROESSE = 7` und passe `ZIEL = (6, 6)` an. Braucht der Agent mehr Trainings-Episoden, um genauso gut zu werden?

3. **Weitblick verändern:** Ändere `weitblick=0.9` in `GitterAgent` auf `weitblick=0.3`. Was passiert mit dem gefundenen Weg? *(Tipp: Ein Agent mit wenig Weitblick "sieht" Belohnungen aus der Ferne kaum noch.)*

4. **Höhere Fallen-Strafe:** Ändere die Strafe für Fallen von `-10` auf `-50`. Macht der Agent jetzt noch größere Umwege, um Fallen zu meiden?

5. **Eigenes Gitter entwerfen:** Verändert Start, Ziel, Fallen und Wand nach eigenen Ideen. Was ist die kürzestmögliche Lösung? Findet der Agent sie?

### 💬 Diskussionsfragen

- Warum kann eine einzelne Aktion ("nach rechts") in einem Feld gut und in einem anderen Feld schlecht sein?
- Was bedeutet es, wenn der **Weitblick** sehr klein ist? Würde der Agent dann noch lernen, Fallen zu meiden, die weit entfernt liegen?
- Wo im echten Leben muss man ähnlich "vorausplanen" statt nur den nächsten Schritt zu bewerten? *(Tipp: Navigationssysteme, Schach, Lagerroboter...)*
- Was wäre der Unterschied, wenn sich die Fallen-Position bei jedem Spiel ändern würde?